# Metro Manila AADT simulation, 2025–2035

Run all cells from top to bottom. This is a **scenario forecast** of daily PCU demand relative to **synthetic, unmeasured capacity** for 20 roads. The Random Forest classifies risk; 4.5% annual growth generates the baseline traffic forecast. A deterministic V/C rule is retained as the reference classification.

## 1. Setup and data checks

Install packages from `requirements.txt` in your Jupyter environment. Open this notebook from the repository root so the relative dataset path works.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from aadt_model import (CLASSES, RISK_ORDER, RING_ROADS, validate_data, add_pcu_and_risk, risk_from_ratio, fit_and_evaluate, simulate, save_outputs)

DATA_PATH = Path('data/metro_manila_aadt_simulation_input_2020_2025.csv')
raw = pd.read_csv(DATA_PATH)
history = add_pcu_and_risk(validate_data(raw))
print(f'{len(history)} historical rows; {history.Road_ID_Name.nunique()} roads; years {history.Year.min()}–{history.Year.max()}')
print('Road types:', history.Road_Type.value_counts().to_dict())
print('Largest absolute PCU recomputation difference:', history.PCU_Difference.abs().max())
assert history.PCU_Difference.abs().max() < 1.0
display(history[['Year', 'Road_ID_Name', 'Road_Type', 'Road_PCU_Volume', 'Capacity_Limit', 'VC_Ratio', 'Rule_Risk_Level']].head())

In [ ]:
for col in ['AADT_Review_Status', 'PCEF_Profile_Status', 'Capacity_Status', 'Data_Flag']:
    print(f'\n{col}:')
    display(history[col].value_counts(dropna=False))
print('2025 baseline risk counts:')
display(history.loc[history.Year.eq(2025), 'Rule_Risk_Level'].value_counts().reindex(RISK_ORDER, fill_value=0))

## 2. Random Forest assessment

Train on 2020–2023 and evaluate on 2024–2025. The target is calculated from PCU/capacity, so this test measures how well the forest reproduces that rule, **not** independent forecasting skill. The direct threshold rule is exact by definition. Forests can also plateau beyond historical feature ranges.

In [ ]:
model, metrics, confusion, comparison = fit_and_evaluate(history)
print('Training classes:', history.loc[history.Year.le(2023), 'Rule_Risk_Level'].value_counts().to_dict())
print('Holdout classes:', comparison.Rule_Risk_Level.value_counts().to_dict())
print('Random Forest holdout metrics:')
display(metrics)
print('Confusion matrix (actual rows, predicted columns):')
display(confusion)
print('Forest disagreements with the direct V/C rule:', int(comparison.Disagreement.sum()), 'of', len(comparison))
display(comparison.loc[comparison.Disagreement].head(20))

## 3. Annual simulation

The 2025 rows are recorded unchanged. For 2026–2035, vehicle counts grow once per year and PCU is recalculated. Baseline capacity stays fixed. The transit scenario moves 20% of projected cars to public buses at an illustrative 40 shifted cars per additional bus; this is a sensitivity assumption, not an observed mode shift. The expansion scenario increases synthetic capacity by 20% on EDSA and C-5 from 2028.

In [ ]:
baseline = simulate(history, model, 'Baseline 4.5%', growth_rate=0.045)
accelerated = simulate(history, model, 'Accelerated 7%', growth_rate=0.07)
transit = simulate(history, model, 'Transit shift 20%', growth_rate=0.045, transfer_share=0.20)
expansion = simulate(history, model, 'EDSA/C-5 expansion 20%', growth_rate=0.045, capacity_increase=0.20)
results = pd.concat([baseline, accelerated, transit, expansion], ignore_index=True)
print('Rows by scenario:', results.groupby('Scenario').size().to_dict())
display(baseline[['Year', 'Road_ID_Name', 'Road_PCU_Volume', 'Capacity_Limit', 'VC_Ratio', 'Risk_Level', 'RF_Risk_Level', 'First_High_Year']].head(20))

In [ ]:
assert risk_from_ratio(0.6999) == 'Low'
assert risk_from_ratio(0.70) == 'Medium'
assert risk_from_ratio(0.95) == 'High'
for scenario, frame in results.groupby('Scenario'):
    assert frame.groupby('Year').size().eq(20).all(), scenario
    assert not frame.duplicated(['Year', 'Road_ID_Name']).any(), scenario
    assert set(frame.Year) == set(range(2025, 2036)), scenario
assert np.allclose(baseline.loc[baseline.Year.eq(2025), 'Road_PCU_Volume'].to_numpy(), history.loc[history.Year.eq(2025), 'Road_PCU_Volume'].to_numpy())
check = baseline.pivot(index='Road_ID_Name', columns='Year', values='Car')
assert np.allclose(check[2026], check[2025] * 1.045)
assert np.allclose(check[2035], check[2025] * 1.045**10)
assert baseline.groupby('Road_ID_Name').Capacity_Limit.nunique().eq(1).all()
print('Simulation checks passed.')

## 4. Results

`First_High_Year` means the first year that the *scenario V/C rule* reaches 0.95 within 2025–2035. A blank value means it did not cross during this horizon. It does not establish the year of physical failure.

In [ ]:
summary = results.loc[results.Year.eq(2035), ['Scenario', 'Road_ID_Name', 'Road_Type', 'Road_PCU_Volume', 'Capacity_Limit', 'VC_Ratio', 'Risk_Level', 'RF_Risk_Level', 'First_High_Year']]
display(summary.sort_values(['Scenario', 'VC_Ratio'], ascending=[True, False]))
risk_counts = results.groupby(['Scenario', 'Year', 'Risk_Level']).size().unstack(fill_value=0).reindex(columns=RISK_ORDER, fill_value=0)
display(risk_counts)
print('RF/rule disagreement by scenario:', results.groupby('Scenario').RF_Disagreement.sum().to_dict())

In [ ]:
risk_numeric = baseline.pivot(index='Road_ID_Name', columns='Year', values='VC_Ratio')
plt.figure(figsize=(13, 8))
sns.heatmap(risk_numeric, cmap='YlOrRd', annot=True, fmt='.2f', cbar_kws={'label':'V/C ratio'})
plt.title('Baseline V/C ratio by road and year — synthetic capacity')
plt.tight_layout()
plt.show()

In [ ]:
type_year = baseline.groupby(['Year', 'Road_Type'], as_index=False).VC_Ratio.median()
plt.figure(figsize=(10, 5))
sns.lineplot(data=type_year, x='Year', y='VC_Ratio', hue='Road_Type', marker='o')
plt.axhline(0.70, ls='--', color='orange', label='Medium threshold')
plt.axhline(0.95, ls='--', color='red', label='High threshold')
plt.title('Median baseline V/C by road type — synthetic capacity')
plt.ylabel('Median V/C ratio')
plt.tight_layout()
plt.show()

In [ ]:
save_outputs(results, Path('outputs'))
print('Saved outputs/road_year_forecast.csv and outputs/scenario_2035_summary.csv')

## Interpretation limits

All supplied AADT source rows require verification, PCU factors are marked as project/literature assumptions, and capacity is explicitly synthetic. The model assumes uniform vehicle-class growth in its baseline; annual averages cannot describe rush-hour queues, incidents, flooding, or vehicle-level behavior. Different scenario results show sensitivity to assumptions, not a calibrated uncertainty interval. Human traffic planners should verify sources and capacity before using these results for decisions.